In [1]:
from datetime import datetime, timedelta
from typing import List, Optional
from pydantic import BaseModel, Field
import math
from collections import Counter

# ==========================================
# 1. КОНСТАНТЫ И НАСТРОЙКИ СИСТЕМЫ
# ==========================================

INTERACTION_WEIGHTS = {
    "coffee_meeting": 5.0,
    "email_thread": 3.0,
    "social_like": 1.0
}

TAG_WEIGHTS = {
    "mentor": 20.0,
    "recruiter": 15.0,
    "peer": 5.0,
    "bridge_contact": 25.0
}

# Настройки для анализа сети
MIN_NETWORK_SIZE = 5 # Минимальный порог для "здоровой" сети

# ==========================================
# 2. МОДЕЛИ ДАННЫХ (Pydantic)
# ==========================================

class Interaction(BaseModel):
    type: str
    date: datetime
    notes: Optional[str] = None

class Contact(BaseModel):
    id: str
    name: str
    industry: str # Обязательное поле для расчета Индекса разнообразия
    tags: List[str] = Field(default_factory=list)
    interactions: List[Interaction] = Field(default_factory=list)
    created_at: datetime = Field(default_factory=datetime.now)

    # Параметры математической модели
    decay_lambda: float = 0.05
    dormant_threshold: float = 10.0

    @property
    def base_weight(self) -> float:
        """Расчет альфы (стартового веса) на основе тегов."""
        return sum(TAG_WEIGHTS.get(tag, 1.0) for tag in self.tags)

    def calculate_current_value(self) -> float:
        """Расчет динамического Индекса силы отношений (с учетом затухания)."""
        total_value = self.base_weight
        now = datetime.now()

        for interaction in self.interactions:
            days_passed = max(0, (now - interaction.date).days)
            weight = INTERACTION_WEIGHTS.get(interaction.type, 1.0)

            decayed_weight = weight * math.exp(-self.decay_lambda * days_passed)
            total_value += decayed_weight

        return round(total_value, 2)

    def get_status(self) -> str:
        """Определение воронки нетворкинга (Target, Active, Dormant)."""
        if not self.interactions:
            return "Target"

        if self.calculate_current_value() < self.dormant_threshold:
            return "Dormant"

        return "Active"


# ==========================================
# 3. АНАЛИЗАТОР СЕТИ (Глобальные метрики)
# ==========================================

class NetworkAnalyzer:
    """Класс для анализа всей сети контактов пользователя."""

    def __init__(self, contacts: List[Contact]):
        self.contacts = contacts

    def calculate_diversity_index(self) -> float:
        """Расчет Индекса разнообразия (Энтропия Шеннона)."""
        if not self.contacts:
            return 0.0

        industries = [contact.industry for contact in self.contacts]
        counts = Counter(industries)
        total_contacts = len(self.contacts)
        S = len(counts)

        if S <= 1:
            return 0.0

        entropy = 0.0
        for count in counts.values():
            p_i = count / total_contacts
            entropy -= p_i * math.log(p_i)

        max_entropy = math.log(S)
        normalized_index = (entropy / max_entropy) * 100

        return round(normalized_index, 2)

    def generate_dashboard_report(self):
        """Генерация сводного отчета для дашборда пользователя."""
        total_contacts = len(self.contacts)

        print("\n=== ДАШБОРД NETWORK PILOT ===")
        print(f"Всего контактов в базе: {total_contacts}")

        # Интеллектуальная подсказка о размере сети
        if total_contacts < MIN_NETWORK_SIZE:
            print("⚠️ [AI-Совет]: Ваша сеть контактов пока слишком мала (менее 5 человек). "
                  "Рекомендуем посетить ближайшие хакатоны или митапы, чтобы расширить круг знакомств!")

        diversity = self.calculate_diversity_index()
        print(f"Индекс разнообразия сети: {diversity} / 100")

        print("\n--- Детализация по контактам ---")
        for contact in self.contacts:
            val = contact.calculate_current_value()
            stat = contact.get_status()
            print(f"👤 {contact.name} | Сфера: {contact.industry} | Ценность: {val} | Статус: {stat}")
        print("===============================\n")


# ==========================================
# 4. ПРИМЕР ИСПОЛЬЗОВАНИЯ В ПРИЛОЖЕНИИ
# ==========================================

# Создаем небольшую базу контактов пользователя (4 человека)
my_contacts = [
    Contact(
        id="1", name="Алексей (Tech Lead)", industry="IT", tags=["mentor", "bridge_contact"]
    ),
    Contact(
        id="2", name="Мария (Junior Dev)", industry="IT", tags=["peer"]
    ),
    Contact(
        id="3", name="Иван (Студент)", industry="IT", tags=["peer"]
    ),
    Contact(
        id="4", name="Ольга (Дизайнер)", industry="Design", tags=["peer"]
    )
]

# Логируем встречи для демонстрации статусов
# С Алексеем встречались месяц назад (будет затухание)
my_contacts[0].interactions.append(
    Interaction(type="coffee_meeting", date=datetime.now() - timedelta(days=30))
)
# С Марией переписывались вчера (высокая актуальность)
my_contacts[1].interactions.append(
    Interaction(type="email_thread", date=datetime.now() - timedelta(days=1))
)
# С Иваном и Ольгой пока не общались (статус Target)

# Инициализируем анализатор и выводим результат
analyzer = NetworkAnalyzer(my_contacts)
analyzer.generate_dashboard_report()


=== ДАШБОРД NETWORK PILOT ===
Всего контактов в базе: 4
⚠️ [AI-Совет]: Ваша сеть контактов пока слишком мала (менее 5 человек). Рекомендуем посетить ближайшие хакатоны или митапы, чтобы расширить круг знакомств!
Индекс разнообразия сети: 81.13 / 100

--- Детализация по контактам ---
👤 Алексей (Tech Lead) | Сфера: IT | Ценность: 46.12 | Статус: Active
👤 Мария (Junior Dev) | Сфера: IT | Ценность: 7.85 | Статус: Dormant
👤 Иван (Студент) | Сфера: IT | Ценность: 5.0 | Статус: Target
👤 Ольга (Дизайнер) | Сфера: Design | Ценность: 5.0 | Статус: Target

